# NPCRA basics — what the metrics actually mean

Pedagogical walkthrough of the indicators implemented in `n24sal.npcra` using **synthetic data** with a known intrinsic period. No personal data is loaded here ; everything is reproducible from seeds.

Two synthetic series are compared throughout:

- **Entrained** : cosinor of period exactly 24h, plus noise
- **Free-running N24** : cosinor of period 24.7h (typical sighted N24 case), plus noise — drifts by ~42 min per calendar day

Each NPCRA indicator is computed on both, with the expected qualitative pattern noted in the prose.

In [1]:
import numpy as np
import plotly.io as pio

from n24sal.npcra import (
    bootstrap_tau_ci,
    circadian_function_index,
    estimate_tau,
    interdaily_stability,
    intradaily_variability,
    l5_m10,
    relative_amplitude,
)
from n24sal.synthetic import generate_synthetic_actigraphy
from n24sal.viz import apply_theme, average_24h_profile, m10_phase_drift_plot

apply_theme("dark")
pio.renderers.default = "notebook_connected"

EPOCHS_PER_HOUR = 60
EPOCHS_PER_DAY = 1440
N_DAYS = 21

## 1. Generate the two reference signals

Both signals share the same noise seed, amplitude and baseline — only the intrinsic period `tau` differs.

In [2]:
entrained = generate_synthetic_actigraphy(n_days=N_DAYS, tau_hours=24.0, noise_sd=5.0, seed=1)
n24 = generate_synthetic_actigraphy(n_days=N_DAYS, tau_hours=24.7, noise_sd=5.0, seed=1)
print(f"entrained: {len(entrained):,} epochs over {N_DAYS} days")
print(f"N24 (tau=24.7h): {len(n24):,} epochs over {N_DAYS} days")

entrained: 30,240 epochs over 21 days
N24 (tau=24.7h): 30,240 epochs over 21 days


## 2. Interdaily Stability (IS) and Intradaily Variability (IV)

**IS** measures how reproducible the 24h profile is across days. Bounded in `[0, 1]`. High IS means the rhythm is locked to the calendar. **An N24 patient should show low IS** because activity drifts across calendar hours.

**IV** measures how fragmented the rhythm is at the epoch scale (frequent rest↔activity transitions). Higher IV = more fragmented sleep / activity bouts.

In [3]:
for label, df in [("entrained (tau=24.0h)", entrained), ("N24 (tau=24.7h)", n24)]:
    arr = df["activity"].to_numpy()
    is_val = interdaily_stability(arr, EPOCHS_PER_DAY)
    iv_val = intradaily_variability(arr)
    print(f"{label:30s}  IS = {is_val:.3f}   IV = {iv_val:.3f}")

entrained (tau=24.0h)           IS = 0.985   IV = 0.033
N24 (tau=24.7h)                 IS = 0.226   IV = 0.033


## 3. L5, M10, Relative Amplitude (RA), CFI

**L5** = mean of the least-active 5h-consecutive window on the average 24h profile. **M10** = the most-active 10h-consecutive window.

**RA = (M10 − L5) / (M10 + L5)** — bounded `[0, 1]`. High RA = strong rest/active contrast = healthy rhythm.

**CFI (Circadian Function Index, Ortiz-Tudela 2010)** combines IS, normalized IV and RA into a single `[0, 1]` health score.

In [4]:
for label, df in [("entrained", entrained), ("N24", n24)]:
    arr = df["activity"].to_numpy()
    res = l5_m10(arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
    ra = relative_amplitude(res.l5_value, res.m10_value)
    cfi = circadian_function_index(
        interdaily_stability(arr, EPOCHS_PER_DAY),
        intradaily_variability(arr),
        ra,
    )
    print(
        f"{label:10s}  L5={res.l5_value:6.2f} @ {res.l5_phase_hours:5.2f}h   "
        f"M10={res.m10_value:6.2f} @ {res.m10_phase_hours:5.2f}h   "
        f"RA={ra:.3f}   CFI={cfi:.3f}"
    )

entrained   L5= 19.85 @  9.03h   M10= 93.76 @ 19.00h   RA=0.651   CFI=0.873
N24         L5= 25.36 @ 16.87h   M10= 70.38 @  2.05h   RA=0.470   CFI=0.560


## 4. The 24h profile, visualised

For the entrained signal the profile should show a clean diurnal peak. For N24 the profile flattens out because the activity peak rotates across calendar hours.

In [5]:
average_24h_profile(entrained, timezone="UTC", title="Average 24h profile — entrained tau=24.0h").show()

In [6]:
average_24h_profile(n24, timezone="UTC", title="Average 24h profile — N24 tau=24.7h").show()

## 5. Estimating tau on the N24 signal

`estimate_tau` fits a line to the daily M10 phase (after circular unwrap). Slope = drift per day in hours ; tau = 24h + slope.

`bootstrap_tau_ci` resamples the daily (day-index, phase) pairs to produce a percentile-based 95% confidence interval on the slope.

In [7]:
n24_arr = n24["activity"].to_numpy()
point = estimate_tau(n24_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
ci = bootstrap_tau_ci(n24_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY, n_iter=1000, seed=1)

print(f"Point estimate: tau = {point.tau_hours:.3f} h")
print(f"R^2 of M10-phase regression: {point.r_squared:.3f}")
print(f"Bootstrap 95% CI: [{ci.ci_low_hours:.3f}, {ci.ci_high_hours:.3f}] over {ci.n_iterations} resamples")
print(f"True tau (synthetic): 24.700 h")

Point estimate: tau = 24.678 h
R^2 of M10-phase regression: 0.999
Bootstrap 95% CI: [24.671, 24.684] over 1000 resamples
True tau (synthetic): 24.700 h


In [8]:
m10_phase_drift_plot(n24, timezone="UTC", title="M10 phase drift — synthetic N24 tau=24.7h").show()

## 6. Comparison with published norms

Values from `data/reference/norms.json` (Van Someren 1999, Ortiz-Tudela 2010, Witting 1990, Sack 2007, Hayakawa 2005).

In [9]:
import json
from pathlib import Path

norms = json.loads(Path("../data/reference/norms.json").read_text())
healthy = norms["healthy_adults"]
n24_pub = norms["N24_published_cases"]

n24_is = interdaily_stability(n24_arr, EPOCHS_PER_DAY)
n24_res = l5_m10(n24_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
n24_ra = relative_amplitude(n24_res.l5_value, n24_res.m10_value)

print("               | this synthetic N24 |  healthy (lit.)  |   N24 (lit.)")
print("---------------|--------------------|------------------|---------------")
print(f"IS             |   {n24_is:.3f}            |   {healthy['IS']['mean']:.2f} ± {healthy['IS']['sd']:.2f}   |   < {n24_pub['IS_typical']['threshold']}")
print(f"RA             |   {n24_ra:.3f}            |   {healthy['RA']['mean']:.2f} ± {healthy['RA']['sd']:.2f}   |   (lower)")
print(f"tau (hours)    |   {point.tau_hours:.3f}            |   24.000        |   {n24_pub['tau_hours']['range_min']}–{n24_pub['tau_hours']['range_max']}")

               | this synthetic N24 |  healthy (lit.)  |   N24 (lit.)
---------------|--------------------|------------------|---------------
IS             |   0.226            |   0.62 ± 0.10   |   < 0.2
RA             |   0.470            |   0.93 ± 0.05   |   (lower)
tau (hours)    |   24.678            |   24.000        |   24.2–25.5


## Key results — synthetic walkthrough

- **IS** sharply discriminates entrained from N24 even on 21-day synthetic data: entrained sits comfortably above the healthy reference distribution while the N24 signal collapses well below.
- **RA** is also lower on the N24 signal but less dramatically (the underlying cosinor still has clean amplitude — it's the *calendar alignment* that breaks).
- **tau** is recovered to within ~0.05h of truth on 21 days at moderate noise. The bootstrap CI quantifies the precision of that recovery ; expect it to tighten by ~1/√n as recording length grows.
- The drift plot makes the methodology visually obvious — the slope IS the chronobiological signature.

These results validate the `n24sal.npcra` implementation against its specification. The next notebook (`03_personal_case.ipynb`) applies the same pipeline to real Samsung Galaxy Watch data.